In [1]:
import sys
import os
sys.path.append(os.path.abspath("../../../src"))

In [2]:
import pandas as pd
df = pd.read_csv("../../../data/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
from utils.preprocess import create_features
df = create_features(df)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0,5.014760,0,1,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,0,1.305626,0,1,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,5.939276,0,1,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0,4.824306,0,1,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,0,4.262539,0,1,0


In [4]:
#Tách Feature và Label
X = df.drop(["Class", "Time"], axis=1).columns.to_list()
y = "Class"

In [5]:
from utils import *
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, X, y)

X_train: (181584, 33) y_train: (181584,)
X_val: (45396, 33) y_val: (45396,)
X_test: (56746, 33) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


In [6]:
#Dùng Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])

pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not wo

In [7]:
pipe.named_steps["model"].n_iter_

array([35], dtype=int32)

In [8]:
#Predict xác suất
y_val_prob = pipe.predict_proba(X_val)[:, 1]
y_test_prob = pipe.predict_proba(X_test)[:, 1]
print(y_val_prob)
print(y_test_prob)

[0.01317983 0.10692782 0.05534639 ... 0.00027914 0.02447189 0.03172686]
[0.020414   0.01766961 0.01418569 ... 0.02209851 0.0777073  0.14370899]


In [9]:
import numpy as np

y_test = np.array(y_test).reshape(-1)
y_test_prob = np.array(y_test_prob).reshape(-1)

print(y_test.shape, y_test_prob.shape)

(56746,) (56746,)


In [10]:
def fast_sweep(y_test, y_test_prob, FP=5.0, FN=200.0):
    idx = np.argsort(-y_test_prob)
    y_test_prob = y_test_prob[idx]
    y_test = y_test[idx]

    tp = np.cumsum(y_test == 1)
    fp = np.cumsum(y_test == 0)

    total_pos = np.sum(y_test == 1)
    total_neg = np.sum(y_test == 0)

    fn = total_pos - tp
    tn = total_neg - fp

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    cost = fp * FP + fn * FN

    df = pd.DataFrame({
        'threshold': y_test_prob,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'cost': cost
    })

    return df.drop_duplicates('threshold')

In [11]:
df_sweep = fast_sweep(y_test, y_test_prob)

df_zoom = df_sweep[(df_sweep['threshold'] >= 0.968) & (df_sweep['threshold'] <= 0.970)]
print(df_zoom)

     threshold  precision    recall        f1  tp     tn   fp  fn    cost
173   0.969935   0.350575  0.824324  0.491935  61  56559  113  13  3165.0
174   0.969543   0.348571  0.824324  0.489960  61  56558  114  13  3170.0
175   0.969542   0.346591  0.824324  0.488000  61  56557  115  13  3175.0
176   0.969133   0.344633  0.824324  0.486056  61  56556  116  13  3180.0
177   0.969047   0.348315  0.837838  0.492063  62  56556  116  12  2980.0
178   0.968505   0.346369  0.837838  0.490119  62  56555  117  12  2985.0
179   0.968155   0.344444  0.837838  0.488189  62  56554  118  12  2990.0
180   0.968043   0.342541  0.837838  0.486275  62  56553  119  12  2995.0


In [12]:
def sweep_zoom_2(y_test, y_test_prob, FP=5.0, FN=200.0):
    thresholds = np.linspace(0.968, 0.970, 2000)  # 👈 thêm ở đây
    
    rows = []
    for thr in thresholds:
        y_pred = (y_test_prob >= thr).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()

        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        cost = fp * FP + fn * FN

        rows.append({
            'threshold': thr,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'tp': tp,
            'tn': tn,
            'fp': fp,
            'fn': fn,
            'cost': cost
        })

    return pd.DataFrame(rows)

In [13]:
df_zoom_2 = sweep_zoom_2(y_test, y_test_prob)
print(df_zoom_2)

      threshold  precision    recall        f1  tp     tn   fp  fn    cost
0      0.968000   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
1      0.968001   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
2      0.968002   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
3      0.968003   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
4      0.968004   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
...         ...        ...       ...       ...  ..    ...  ...  ..     ...
1995   0.969996   0.352601  0.824324  0.493927  61  56560  112  13  3160.0
1996   0.969997   0.352601  0.824324  0.493927  61  56560  112  13  3160.0
1997   0.969998   0.352601  0.824324  0.493927  61  56560  112  13  3160.0
1998   0.969999   0.352601  0.824324  0.493927  61  56560  112  13  3160.0
1999   0.970000   0.352601  0.824324  0.493927  61  56560  112  13  3160.0

[2000 rows x 9 columns]


In [15]:
import pandas as pd

pd.set_option('display.max_rows', None)
print(df_zoom_2)

      threshold  precision    recall        f1  tp     tn   fp  fn    cost
0      0.968000   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
1      0.968001   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
2      0.968002   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
3      0.968003   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
4      0.968004   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
5      0.968005   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
6      0.968006   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
7      0.968007   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
8      0.968008   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
9      0.968009   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
10     0.968010   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
11     0.968011   0.342541  0.837838  0.486275  62  56553  119  12  2995.0
12     0.968012   0.34254